# 02 — Contrats de données avec Pandera

**Projet** : Prédiction d'attrition client (churn télécom) avec XGBoost
**Objectif** : transformer les règles métier découvertes en EDA en **contrats exécutables**.

Un contrat de données vaut par ses deux propriétés :

1. il **accepte** les données conformes (sinon il bloque la production pour rien) ;
2. il **refuse** les données corrompues avec un message exploitable (sinon il ne sert à rien).

Ce notebook démontre les deux — y compris en **provoquant volontairement** des échecs.

## Objectifs pédagogiques

1. Lire un schéma `DataFrameModel` comme une documentation (types, bornes, catégories, unicité).
1. Provoquer et interpréter un échec de validation (`failure_cases`).
1. Distinguer les trois contrats du cycle de vie : brut, transformé, inférence.
1. Utiliser le mode `lazy` pour remonter toutes les erreurs d'un coup.

**Objectifs transverses du dépôt**

- Comprendre le gradient boosting : arbres séquentiels qui corrigent les résidus, shrinkage et régularisation.
- Utiliser l'early stopping natif (`eval_set` + `early_stopping_rounds`) sans réinventer la boucle d'entraînement.
- Régulariser un booster (max_depth, min_child_weight, subsample, colsample_bytree, reg_lambda, gamma).

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

# --- Racine du projet ---------------------------------------------------------------------------
# Le notebook s'exécute depuis `notebooks/` : on remonte d'un cran pour pouvoir importer `src`.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hydra import compose, initialize_config_dir  # noqa: E402
from hydra.core.global_hydra import GlobalHydra  # noqa: E402
from loguru import logger  # noqa: E402

from src.schemas.config import validate_config  # noqa: E402
from src.utils.paths import ProjectPaths  # noqa: E402

# --- Réglages d'affichage -----------------------------------------------------------------------
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)
logger.remove()
logger.add(sys.stderr, level="WARNING")

# --- Configuration : exactement celle de `python -m src.main` ------------------------------------
# Les notebooks travaillent sur un échantillon réduit (1500 lignes) : l'exécution complète
# reste sous la minute, tout en conservant des distributions réalistes.
NB_ROWS = 1500

GlobalHydra.instance().clear()
with initialize_config_dir(config_dir=str(PROJECT_ROOT / "conf"), version_base=None):
    CONFIG = validate_config(
        compose(
            config_name="config",
            overrides=[
                "mode=train",
                f"data.n_samples={NB_ROWS}",
                "seed=42",
                "log_level=WARNING",
                "++train.epochs=3",
                "train.callbacks.progress_bar=false",
            ],
        )
    )

PATHS = ProjectPaths.from_root(PROJECT_ROOT)
# Les notebooks écrivent leurs artefacts dans `outputs/notebooks` (ignoré par git) afin de ne
# jamais écraser ceux produits par `make train`.
NB_PATHS = ProjectPaths.from_root(PROJECT_ROOT / "outputs" / "notebooks").ensure()

print(f"Projet            : {CONFIG.project.name}")
print(f"Tâche             : {CONFIG.metrics.task}")
print(f"Métrique primaire : {CONFIG.metrics.primary} (seuil cible : 0.7)")
print(f"Cible             : {CONFIG.data.target}")
print(f"Algorithme        : {CONFIG.model.algorithm} ({CONFIG.model.name})")
print(f"Lignes (notebook) : {NB_ROWS}")

In [ ]:
from src.data.generators import SyntheticDataGenerator
from src.data.loaders import RawDataLoader

raw_path = PATHS.data_file(CONFIG.data.dataset_name)
if raw_path.exists():
    # Cas nominal : le dataset a été généré par `make data`, on passe par le loader validant.
    raw = RawDataLoader(PATHS, dataset_name=CONFIG.data.dataset_name).load()
    print(f"Dataset lu depuis {raw_path.relative_to(PROJECT_ROOT)}")
else:
    # Le notebook reste exécutable sur un clone frais : on génère en mémoire.
    raw = SyntheticDataGenerator(n_samples=NB_ROWS, seed=CONFIG.data.seed).generate()
    print("data/raw vide : génération synthétique en mémoire (`make data` la persiste)")

raw = raw.head(NB_ROWS).reset_index(drop=True)
print(f"shape = {raw.shape}")
raw.head()

**Ce qu'il faut retenir**

- Le `RawDataLoader` applique déjà le contrat au chargement : une source déviante échoue **ici**, pas en entraînement.
- `validate=False` existe pour inspecter des données cassées sans exception (diagnostic).

## 1. Le contrat des données brutes

Trois schémas cohabitent dans `src/data/schemas.py` :

| Schéma | Appliqué sur | Politique |
| --- | --- | --- |
| `RawDataSchema` | `data/raw` juste après chargement | strict, types et bornes imposés |
| `ProcessedDataSchema` | matrice livrée au modèle | 100 % numérique, zéro NaN |
| `InferenceDataSchema` | toute requête de prédiction | colonnes optionnelles, nulls tolérés |

In [ ]:
from src.data.schemas import RawDataSchema, schema_to_markdown

display(Markdown(schema_to_markdown("raw")))

**Ce qu'il faut retenir**

- Cette table est **générée depuis le code** : elle ne peut pas diverger de l'implémentation.
- `nullable=False` sur la clé et `unique` garantissent l'intégrité du jeu (pas de doublon silencieux).
- Les `checks` (bornes, `isin`) sont la traduction directe des règles métier du README.

In [ ]:
from src.data.schemas import describe_schema

describe_schema("raw")

## 2. Validation nominale : le contrat doit passer

In [ ]:
from src.data.schemas import validate_frame, validation_report

validated = validate_frame(raw, "raw")
profile = validation_report(validated)
print(f"validation OK | lignes={profile['n_rows']} | colonnes={profile['n_columns']}")
print(f"              | cellules manquantes={profile['missing_cells']}")
validated.head(3)

**Ce qu'il faut retenir**

- La coercition (`coerce=True`) absorbe les différences Parquet/CSV : un entier lu comme flottant reste valide.
- Un contrat qui ne passe **jamais** en local est un contrat mal calibré — le vérifier fait partie du travail.

## 3. Échecs volontaires — la partie la plus utile du notebook

On corrompt **délibérément** le dataset pour vérifier que le contrat mord. Chaque cellule
isole une violation ; l'exception est capturée puis affichée avec ses `failure_cases`.

In [ ]:
try:
    import pandera.pandas as pa
except ModuleNotFoundError:  # pandera < 0.26
    import pandera as pa

SchemaViolation = (pa.errors.SchemaError, pa.errors.SchemaErrors)


def show_violation(label: str, frame: pd.DataFrame) -> None:
    """Validate a deliberately corrupted frame and explain the failure.

    Args:
        label: Human readable name of the injected corruption.
        frame: Corrupted dataset.
    """
    try:
        RawDataSchema.validate(frame, lazy=True)
    except SchemaViolation as error:
        cases = getattr(error, "failure_cases", None)
        print(f"[REFUSÉ] {label}")
        if cases is not None:
            print(cases.head(6).to_string(index=False))
        else:
            print(error)
        return
    print(f"[ACCEPTÉ — ATTENTION] {label} : le contrat ne couvre pas ce cas")

In [ ]:
numeric_bounded = [
    name
    for name, column in RawDataSchema.to_schema().columns.items()
    if any(
        getattr(check, "name", "") in {"ge", "greater_than_or_equal_to", "in_range"}
        for check in (getattr(column, "checks", []) or [])
    )
]
column = numeric_bounded[0]
corrupted = raw.copy()
corrupted.loc[corrupted.index[:5], column] = 10_000_000
show_violation(f"valeur hors bornes sur `{column}` (10 000 000)", corrupted)

**Ce qu'il faut retenir**

- `failure_cases` donne la **colonne**, le **check** et les **valeurs** en échec : le diagnostic est immédiat.
- En production, cette erreur doit faire échouer le run (fail fast) plutôt que d'entraîner un modèle sur des données fausses.

In [ ]:
corrupted = raw.copy()
corrupted["colonne_non_declaree"] = 0
show_violation("colonne non déclarée (strict=True)", corrupted)

In [ ]:
corrupted = raw.drop(columns=[raw.columns[-1]])
show_violation("colonne manquante", corrupted)

In [ ]:
categorical_checked = [
    name
    for name, column in RawDataSchema.to_schema().columns.items()
    if any(getattr(check, "name", "") == "isin" for check in (getattr(column, "checks", []) or []))
]
if categorical_checked:
    column = categorical_checked[0]
    corrupted = raw.copy()
    corrupted.loc[corrupted.index[:3], column] = "modalite_inexistante"
    show_violation(f"catégorie hors liste sur `{column}`", corrupted)
else:
    print("Aucune colonne contrainte par `isin` dans ce schéma.")

**Ce qu'il faut retenir**

- Une nouvelle modalité non déclarée est le bug silencieux le plus fréquent après un changement de SI amont.
- Deux réponses possibles : mettre à jour le contrat (évolution légitime) ou refuser (régression).

In [ ]:
corrupted = raw.copy()
corrupted.loc[corrupted.index[1], CONFIG.data.id_column] = corrupted.loc[
    corrupted.index[0], CONFIG.data.id_column
]
show_violation(f"clé dupliquée sur `{CONFIG.data.id_column}`", corrupted)

In [ ]:
corrupted = raw.copy()
corrupted.loc[corrupted.index[0], CONFIG.data.id_column] = None
show_violation(f"clé nulle sur `{CONFIG.data.id_column}`", corrupted)

**Ce qu'il faut retenir**

- Une clé dupliquée crée une **fuite** entre splits : la même observation peut se retrouver en train et en test.
- C'est pourquoi `assert_no_overlap()` est testé dans `tests/test_loaders.py`.

## 4. Mode `lazy` : tout remonter d'un coup

In [ ]:
column = numeric_bounded[0]
corrupted = raw.copy()
corrupted.loc[corrupted.index[:20], column] = -1_000_000
corrupted.loc[corrupted.index[20:40], column] = 1_000_000
show_violation(f"40 violations sur `{column}` (mode lazy)", corrupted)

**Ce qu'il faut retenir**

- Sans `lazy`, pandera s'arrête à la première erreur : on découvre les problèmes un par un.
- Avec `lazy`, le rapport complet permet de corriger le flux amont en une fois (configurable via `data.validation.lazy`).

## 5. Contrat des données transformées

In [ ]:
from src.data.schemas import ProcessedDataSchema

try:
    ProcessedDataSchema.validate(raw)
except SchemaViolation as error:
    print("[REFUSÉ comme attendu] la matrice brute n'est pas numérique :")
    print(str(error)[:320])

numeric_matrix = raw.select_dtypes(include=[np.number]).dropna().head(50)
print("matrice numérique valide ->", ProcessedDataSchema.validate(numeric_matrix).shape)

**Ce qu'il faut retenir**

- Ce contrat est la **dernière ligne de défense** avant le modèle : aucune feature texte, aucun NaN, aucun infini.
- Il est appliqué automatiquement par `TrainPipeline` quand `data.validation.processed: true`.

## 6. Contrat d'inférence : tolérant mais pas laxiste

In [ ]:
from src.data.schemas import InferenceDataSchema

payload = (
    raw.drop(columns=[CONFIG.data.target]).head(20).copy()
    if CONFIG.data.target
    else raw.head(20).copy()
)
payload.iloc[0, 0] = None  # valeur manquante tolérée
payload["colonne_du_client"] = "web"  # colonne supplémentaire tolérée
print("payload accepté ->", InferenceDataSchema.validate(payload).shape)

bad_payload = payload.copy()
if numeric_bounded:
    bad_payload[numeric_bounded[0]] = "pas-un-nombre"
try:
    InferenceDataSchema.validate(bad_payload)
except SchemaViolation as error:
    print("[REFUSÉ] type incohérent :", str(error)[:200])

**Ce qu'il faut retenir**

- La cible est absente d'un payload d'inférence : le contrat ne doit pas l'exiger (`required=False`).
- Tolérer les nulls et les colonnes en trop **sans** renoncer aux checks de type : c'est l'équilibre recherché.
- Une requête partielle est corrigée par le preprocessing ; une requête incohérente est refusée avec un message clair.

## 7. Où la validation s'exécute vraiment

| Emplacement | Contrat | Déclencheur |
| --- | --- | --- |
| `RawDataLoader.load()` | `RawDataSchema` | chaque chargement de données |
| `TrainPipeline._preprocess()` | `ProcessedDataSchema` | avant entraînement |
| `Predictor.predict()` | `InferenceDataSchema` | chaque requête de prédiction |
| `tests/test_data_schemas.py` | les trois | chaque commit (`make test`) |

### Checklist à reproduire sur un nouveau projet

1. Écrire le schéma **avant** le code de chargement (le contrat guide l'implémentation).
2. Ajouter un test d'acceptation (données conformes) **et** un test de refus (données corrompues).
3. Versionner le schéma avec le code : une évolution de contrat est un changement d'API.
4. Journaliser les `failure_cases` : ce sont eux qui font gagner du temps en incident.